### Simple RAG using Minsearch by alexeygrigorev

In [33]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/cohorts/2025/01-intro/documents.json

--2026-05-04 20:09:10--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/cohorts/2025/01-intro/documents.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 658332 (643K) [text/plain]
Saving to: ‘documents.json.1’

documents.json.1    100%[===================>] 642.90K  --.-KB/s    in 0.005s  

2026-05-04 20:09:11 (139 MB/s) - ‘documents.json.1’ saved [658332/658332]



In [34]:
!wget https://raw.githubusercontent.com/alexeygrigorev/minsearch/refs/heads/main/minsearch.py

--2026-05-04 20:09:15--  https://raw.githubusercontent.com/alexeygrigorev/minsearch/refs/heads/main/minsearch.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4404 (4.3K) [text/plain]
Saving to: ‘minsearch.py.1’

minsearch.py.1      100%[===================>]   4.30K  --.-KB/s    in 0s      

2026-05-04 20:09:15 (47.5 MB/s) - ‘minsearch.py.1’ saved [4404/4404]



In [35]:
import minsearch
import json

In [36]:
with open('documents.json','rt') as docs:
    docs_raw=json.load(docs)

In [37]:
documents=[]

count=0
for course_dict in docs_raw:
    for docs in course_dict['documents']:
        docs['course']=course_dict['course']
        documents.append(docs)

In [38]:
index=minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

In [39]:
index.fit(documents)

In [40]:
question="How long is this course going to be?"

In [41]:
boost={'question': 3.0, 'section':0.5}

results=index.search(
    query=question,
    boost_dict=boost,
    num_results=5
)

In [42]:
results

[{'text': 'Approximately 4 months, but may take more if you want to do some extra activities (an extra project, an article, etc)',
  'section': 'General course-related questions',
  'question': 'How long is the course?',
  'course': 'machine-learning-zoomcamp'},
 {'text': 'The course videos are pre-recorded, you can start watching the course right now.\nWe will also occasionally have office hours - live sessions where we will answer your questions. The office hours sessions are recorded too.\nYou can see the office hours as well as the pre-recorded course videos in the course playlist on YouTube.',
  'section': 'General course-related questions',
  'question': 'Is it going to be live? When?',
  'course': 'machine-learning-zoomcamp'},
 {'text': 'Each submitted project will be evaluated by 3 (three) randomly assigned students that have also submitted the project.\nYou will also be responsible for grading the projects from 3 fellow students yourself. Please be aware that: not complying to

### Using LLM

In [43]:
from openai import OpenAI

In [45]:
client=OpenAI()

In [46]:
response=client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role":"user","content":question}]
)

In [52]:
response.choices[0].message.content

'I would need more context to answer that question accurately. Could you provide details about the course, such as its name, subject, or format? This would help me give you a more specific answer.'

In [62]:
prompt="""
You are an administrative officer to a course-offering Institution. Your task is to answer questions BASED on the FAQ database.
Only refer to informations provided in CONTEXT when answering the QUESTION. If there are none then provide no answer.

QUESTION: {question}
CONTEXT: {context}

"""

In [65]:
context=""

for doc in results:
    context=context+ f"section: {doc['section']}\n question: {doc['question']}\n answer: {doc['text']}\n\n"

In [66]:
system_prompt=prompt.format(question=question, context=context).strip()

In [67]:
response=client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role":"user","content":system_prompt}]
)

In [68]:
response.choices[0].message.content

'The course is going to be approximately 4 months, but it may take longer if you choose to do some extra activities such as an extra project or an article.'

In [75]:
def search(query):
    boost={'question': 3.0, 'section':0.5}

    results=index.search(
        query=query,
        boost_dict=boost,
        filter_dict={'course':'machine-learning-zoomcamp'},
        num_results=5
    )

    return results

In [82]:
def build_prompt(query,res):
    prompt="""
        You are an administrative officer to a course-offering Institution. Your task is to answer questions BASED on the FAQ database.
    Only refer to informations provided in CONTEXT when answering the QUESTION. If there are none then provide no answer.
    
    QUESTION: {question}
    CONTEXT: {context}

    """.strip()
    context=""

    for doc in res:
        context=context+ f"section: {doc['section']}\n question: {doc['question']}\n answer: {doc['text']}\n\n"
    system_prompt=prompt.format(question=query, context=context).strip()
    return system_prompt

In [83]:
def llm(prompt):
    response=client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role":"user", "content":prompt}]
    )
    return response.choices[0].message.content

In [86]:
def rag(query):
    search_results=search(query)
    prompt=build_prompt(query,search_results)
    answer=llm(prompt)
    return answer

In [87]:
rag(question)

'The course is approximately 4 months long, but it may take longer if you choose to engage in extra activities such as an additional project or article.'

### ElasticSearch

In [89]:
from elasticsearch import Elasticsearch

In [97]:
es_client=Elasticsearch('http://127.0.0.1:9200/')

In [99]:
es_client.info()

ObjectApiResponse({'name': '6fdb188da439', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'P4aln2i-RHWo_RAQelxDRw', 'version': {'number': '9.3.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '17b451d8979a29e31935fe1eb901310350b30e62', 'build_date': '2026-01-29T10:05:46.708397977Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

In [100]:
index_settings={
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

In [102]:
index_name='course-questions'
es_client.indices.create(index=index_name,body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [104]:
from tqdm.auto import tqdm

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [106]:
for i in tqdm(documents):
    es_client.index(index=index_name,document=i)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 948/948 [00:04<00:00, 196.08it/s]


In [130]:
def elastic_search(query):
    search_query={
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }
    response=es_client.search(index=index_name, body=search_query)
    res=[]
    for i in response['hits']['hits']:
        res.append(i['_source'])
    return res
    

In [131]:
elastic_search('just found this course. Any chance i could still join?')

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'After you create a GitHub account, you should clone the course repo to your local machine using the process outlined in this video: Git for Everybody: How to Clone a Repository from GitHub\nHaving this local repository on your computer will make it easy for you to access the instructors’ code and make pull requests (if you want to add your own notes or make changes to the course content).\nYou will probably also create your own repositories that host your notes, versions of your file, to do this. Here is a great tutorial that shows you how to do this: https://www.atlassian.com/git/tutorials/

### RAG using Elastic

In [132]:
# copy previous function
def rag(query):
    search_results=elastic_search(query)
    prompt=build_prompt(query,search_results)
    answer=llm(prompt)
    return answer

In [133]:
rag('just found this course. Any chance i could still join?')

"Yes, even if you don't register, you're still eligible to submit the homeworks. However, be aware that there will be deadlines for turning in the final projects, so don't leave everything for the last minute."